<a href="https://colab.research.google.com/github/SaloneJJ/FHT-Structures-ans-Sperner-Enumerations/blob/main/FHT_Enumeration_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pynauty

In [61]:
"FHT Enumeration by SALONE Jean-Jacques 2026"
"jean-jacques.salone@univ-antilles.fr"

from itertools import combinations
from collections import Counter
import math
from pynauty import Graph, certificate


# ==========================================
# PART 1: Calculation of S(v) using pynauty
# ==========================================

def get_possible_edges(v):
    """Generates all possible non-empty hyperedges as binary integer bitmasks."""
    edges = []
    for r in range(1, v + 1):
        for comb in combinations(range(v), r):
            edge_mask = 0
            for vertex in comb:
                edge_mask |= (1 << vertex)
            edges.append(edge_mask)
    return edges

def is_sperner_extension(current_edges, new_edge):
    """Checks if adding new_edge respects the Sperner condition (clutter property)."""
    for e in current_edges:
        if (new_edge & e) == new_edge or (new_edge & e) == e:
            return False
    return True

def is_connected_bit(v, edges):
    """Verifies the connectedness of the primal graph using bitmask operations."""
    if not edges:
        return False
    adj = [0] * v
    for e in edges:
        verts = []
        temp = e
        vertex_idx = 0
        while temp:
            if temp & 1:
                verts.append(vertex_idx)
            temp >>= 1
            vertex_idx += 1

        for i in range(len(verts)):
            for j in range(i + 1, len(verts)):
                u, w = verts[i], verts[j]
                adj[u] |= (1 << w)
                adj[w] |= (1 << u)

    visited = 1 << 0
    queue = [0]
    while queue:
        curr = queue.pop(0)
        neighbors = adj[curr]
        unvisited = neighbors & ~visited
        while unvisited:
            bit = unvisited & -unvisited
            next_v = bit.bit_length() - 1
            visited |= bit
            queue.append(next_v)
            unvisited ^= bit

    return visited == ((1 << v) - 1)

def get_pynauty_graph(v, edges):
    """Constructs a bipartite graph representing the hypergraph structure for pynauty."""
    m = len(edges)
    n_nodes = v + m
    adj = {i: [] for i in range(n_nodes)}

    vertex_class = set(range(v))
    edge_class = set(range(v, v + m))

    for idx, e in enumerate(edges):
        edge_node = v + idx
        temp = e
        u = 0
        while temp:
            if temp & 1:
                adj[u].append(edge_node)
                adj[edge_node].append(u)
            temp >>= 1
            u += 1

    g = Graph(
        number_of_vertices=n_nodes,
        directed=False,
        adjacency_dict=adj,
        vertex_coloring=[vertex_class, edge_class]
    )
    return g

def compute_S_v(v):
    """Computes S(v) by combining backtracking, clutter, connectedness, and pynauty isomorphism filtering."""
    if v == 1 or v == 2:
        return 1

    all_edges = get_possible_edges(v)
    seen_certificates = set()
    final_classes = set()

    def backtrack(edge_index, current_hg):
        if current_hg:
            g = get_pynauty_graph(v, current_hg)
            cert = certificate(g)
            if cert in seen_certificates:
                return
            seen_certificates.add(cert)

            if is_connected_bit(v, current_hg):
                final_classes.add(cert)

        for i in range(edge_index, len(all_edges)):
            candidate = all_edges[i]
            if is_sperner_extension(current_hg, candidate):
                current_hg.append(candidate)
                backtrack(i + 1, current_hg)
                current_hg.pop()

    backtrack(0, [])
    return len(final_classes)


# ==========================================
# PART 2: Decomposition algorithm for I(v)
# ==========================================

def generate_valid_partitions(v):
    """Generates partitions of v according to the rules: v_1 > 1, v_k > 0, and v_k < v."""
    if v <= 2:
        return []

    partitions = []
    def backtrack_part(remaining, max_val, current):
        if remaining == 0:
            if len(current) > 0 and current[0] > 1:
                partitions.append(tuple(current))
            return
        start = min(remaining, max_val)
        for i in range(start, 0, -1):
            if i < v:
                backtrack_part(remaining - i, i, current + [i])

    backtrack_part(v, v - 1, [])
    seen = set()
    valid = []
    for p in partitions:
        if p[0] > 1 and all(x < v for x in p) and p not in seen:
            seen.add(p)
            valid.append(p)
    return valid

def compute_I_v(v, f_dict):
    """Computes I(v) using partition orbits and multiset combinations of sub-FHTs."""
    if v <= 2:
        return 0

    partitions = generate_valid_partitions(v)
    total_I = 0

    for p in partitions:
        counter = Counter(p)

        # Multiply multiset combinations of sub-FHTs for each block size in the partition
        ways_sub_fhts = 1
        for val, count in counter.items():
            ways_sub_fhts *= math.comb(f_dict[val] + count - 1, count)

        total_I += ways_sub_fhts

    return total_I


# ==========================================
# MAIN SCRIPT
# ==========================================

if __name__ == "__main__":
    try:
        max_v = int(input("Enter the maximum value of v to calculate: "))
        if max_v < 1:
            print("Please enter an integer greater than or equal to 1.")
            exit()
    except ValueError:
        print("Invalid input.")
        exit()

    f = {}
    S = {}
    I = {}

    print(f"\n--- Starting calculations from v = 1 to {max_v} ---\n")

    for v in range(1, max_v + 1):
        # Step 1: Compute S(v)
        if v == 1 or v == 2:
            S[v] = 1
        else:
            print(f"[{v}] Calculating S({v}) using pynauty...")
            S[v] = compute_S_v(v)

        # Step 2: Compute I(v) via isomorphic partition decomposition
        if v == 1 or v == 2:
            I[v] = 0
        else:
            I[v] = compute_I_v(v, f)

        # Step 3: Deduce f(v)
        f[v] = S[v] + I[v]

        print(f"--> Results for v = {v}: S({v}) = {S[v]} | I({v}) = {I[v]} | f({v}) = {f[v]}\n")

Enter the maximum value of v to calculate: 5

--- Starting calculations from v = 1 to 5 ---

--> Results for v = 1: S(1) = 1 | I(1) = 0 | f(1) = 1

--> Results for v = 2: S(2) = 1 | I(2) = 0 | f(2) = 1

[3] Calculating S(3) using pynauty...
--> Results for v = 3: S(3) = 3 | I(3) = 1 | f(3) = 4

[4] Calculating S(4) using pynauty...
--> Results for v = 4: S(4) = 14 | I(4) = 6 | f(4) = 20

[5] Calculating S(5) using pynauty...
--> Results for v = 5: S(5) = 157 | I(5) = 30 | f(5) = 187

